In [ ]:
import os

os.environ["PYTHONHASHSEED"] = "51"

In [ ]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/Colab Notebooks/Applied Computer Vision/Project_Scratchpad Ideas/"

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
# Install dependencies
%%capture
%pip install --no-cache-dir -r requirements.txt

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import cv2
from collections import deque
import numpy as np
import fiftyone as fo
import uuid
import faiss

In [ ]:
from config import NUM_WORKERS, DEVICE, SEED
from utils import ensure_dir
from FiftyOne import FODataset

In [ ]:
DATASET_NAME = "jaguar_stage1.5_deduplication"  
BACKBONE = "Hybrid_Dino_MegaDescriptor"
FODATASET_DIR_NAME = "jaguar_export"

In [ ]:
# set paths
DRIVE_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Applied Computer Vision/Project_Scratchpad Ideas")
WORK_ROOT = Path("/content")

DATA_ROOT = DRIVE_ROOT / f"datasets/{FODATASET_DIR_NAME}"
EMBS_DIR = DATA_ROOT / "embeddings"
EXPORT_DIR = DRIVE_ROOT / f"datasets/{DATASET_NAME}_metadata"

OUT_DIR = WORK_ROOT / "out"

ensure_dir([OUT_DIR, EXPORT_DIR, EMBS_DIR])

In [ ]:
# duplication from notebook stage 01
# Load specific embeddings
def load_embeddings(model_name, manipulation_mode, save_dir):
    """Loads specific embeddings and their corresponding identifiers."""
    save_dir = Path(save_dir)
    
    emb_file = save_dir / f"embeddings_{model_name}_{manipulation_mode}.npy"
    if not emb_file.exists():
        print(f"❌ Error: Could not find {emb_file}")
        return None
    
    # These usually stay the same across runs if the dataset didn't change          !!TODO: check than what happens with different manipulation modes
    ids_file = save_dir / "sample_ids.npy"
    fps_file = save_dir / "sample_filepaths.npy" 

    # Load the files
    # allow_pickle=True is necessary because sample_ids and filepaths are stored as Python 'object' arrays (strings/IDs)
    embs = np.load(emb_file)
    ids = np.load(ids_file, allow_pickle=True)
    filepaths = np.load(fps_file, allow_pickle=True)
    
    print(f"✅ Loaded {model_name} ({manipulation_mode}): {embs.shape}")

    return embs, ids, filepaths

In [ ]:
manipulation_mode = "original"

embs, ids, filepaths = load_embeddings(BACKBONE, manipulation_mode, EMBS_DIR)

absolute_filepaths = [str(DATA_ROOT / f) for f in filepaths]

In [ ]:
fo_dataset = FODataset.load_from_export(
    export_dir= DATA_ROOT,
    dataset_name=DATASET_NAME
)
fo_ds = fo_dataset.get_dataset()

session = fo_dataset.launch()

In [ ]:
print(session.url)

print(fo_ds, fo_ds.count())
print(fo_ds.get_field_schema().keys())

labels = [s.get("label") for s in fo_ds.samples]
# labels = [s.get("ground_truth").get("label") for s in dataset.samples]
fps = [s.get("filepath") for s in fo_ds.samples]

## 1) Find Bursts

How: Group images into a "Burst" (Identity is the same + Similarity > 0.98
Action: Keep one representative (center of centroid) with highest resolution and sharpness

How: 
1) For each group, calculate the mean embedding.
2) Find the single image in that group whose embedding is closest (Cosine Similarity) to the mean. 
Finding Resolution/Sharpness:
Resolution: Simply width * height.
Sharpness: Use the Laplacian Variance method. It’s a standard computer vision trick.


3) Rest: Tag as is_duplicate = True 

In [ ]:
def get_connected_clusters(sims, idxs, threshold=0.95):
    """
    Builds a graph where edges exist if similarity >= threshold,
    then finds connected components.
    """
    N = sims.shape[0]
    adj = [[] for _ in range(N)]
    
    # Build adjacency list only for similarities above threshold
    # Start from index 1 to skip 'self'
    for i in range(N):
        for j, sim in zip(idxs[i, 1:], sims[i, 1:]):
            if sim >= threshold:
                adj[i].append(int(j))
                adj[int(j)].append(i)

    # Connected components using BFS
    comp_id = -np.ones(N, dtype=int)
    clusters = []
    cid = 0

    for i in range(N):
        if comp_id[i] != -1:
            continue
        
        q = deque([i])
        comp_id[i] = cid
        nodes = [i]
        
        while q:
            u = q.popleft()
            for v in adj[u]:
                if comp_id[v] == -1:
                    comp_id[v] = cid
                    q.append(v)
                    nodes.append(v)
        
        if len(nodes) >= 2:  # Only keep non-trivial clusters (size>=2)
            clusters.append(nodes)
        cid += 1
        
    return clusters

In [ ]:
# duplication from notebook 2
def find_nearest_neighbors(embeddings, k=100):           # neighbors (incl self)
    """
    Performs FAISS Inner Product search on L2-normalized embeddings.
    Returns:
        sims: (N, k) similarity scores (0.0 to 1.0)
        idxs: (N, k) indices of neighbors
    """
    # embs: (N, D) float32, L2-normalized
    embs = embeddings.astype("float32")
    # L2 Normalization makes Inner Product equivalent to Cosine Similarity
    norms = np.linalg.norm(embs, axis=1, keepdims=True)
    embs /= (norms + 1e-12)
    
    D = embs.shape[1]
    index = faiss.IndexFlatIP(D)        # inner product
    index.add(embs)                     # add N vectors
    
    sims, idxs = index.search(embs, k)  # sims: (N,k), idxs: (N,k)
    return sims, idxs

In [ ]:
# Hyperparameters:
k = 50
similarity_threshold = 0.856     # 0.90 finds 676 bursts/duplicates.  |. 0.87 finds 735 bursts/duplicates. | sweet spot: 0.856 finds 761 duplicates and 170 potential keeps

# Find clusters
sims, idxs = find_nearest_neighbors(embs, k=k)
clusters = get_connected_clusters(sims, idxs, threshold=similarity_threshold)

# Lookup: Map indices back to filenames          ## !! TODO:Test
clusters_as_filepaths = []
for cluster in clusters:
    group = [absolute_filepaths[i] for i in cluster] # or use all_ids[i]
    clusters_as_filepaths.append(group)

print(f"Found {len(clusters_as_filepaths)} groups of duplicates.")

In [ ]:
import numpy as np
from collections import deque

def has_cross_label_mismatch(sims, idxs, labels, threshold):
    """
    Returns True if any cluster created by this threshold contains
    multiple different labels.
    Finds thereby lower threshold for similarity. 
    """
    N = sims.shape[0]
    adj = [[] for _ in range(N)]
    for i in range(N):
        # We only look at neighbors that are within our pre-computed k
        for j_idx, sim in enumerate(sims[i]):
            if sim >= threshold:
                v = idxs[i, j_idx]
                if i != v:
                    adj[i].append(int(v))
                    adj[int(v)].append(i)

    visited = np.zeros(N, dtype=bool)
    for i in range(N):
        if not visited[i]:
            # BFS to find cluster
            cluster_indices = []
            q = deque([i])
            visited[i] = True
            while q:
                u = q.popleft()
                cluster_indices.append(u)
                for v in adj[u]:
                    if not visited[v]:
                        visited[v] = True
                        q.append(v)

            # Check labels in this cluster
            if len(cluster_indices) > 1:
                cluster_labels = {labels[idx] for idx in cluster_indices}
                if len(cluster_labels) > 1:
                    return True # FOUND A MISMATCH
    return False

def find_optimal_threshold(sims, idxs, labels, low=0.8, high=1.0, tolerance=0.001):
    """
    Binary search for the lowest possible threshold with zero label mismatches.
    """
    best_safe_t = high

    print(f"Starting Binary Search for optimal threshold in range [{low}, {high}]...")

    while (high - low) > tolerance:
        mid = (low + high) / 2

        if has_cross_label_mismatch(sims, idxs, labels, mid):
            # Mid is too low (aggressive), caused a collision
            low = mid
        else:
            # Mid is safe, let's try to go even lower
            best_safe_t = mid
            high = mid

        print(f"  Tested T={mid:.4f} | Result: {'❌ Mismatch' if mid == low else '✅ Clean'}")

    print(f"\n🎯 Sweet Spot Threshold: {best_safe_t:.4f}")
    return best_safe_t

# --- EXECUTION ---
sweet_spot_t = find_optimal_threshold(sims, idxs, labels)

In [ ]:
def calculate_sharpness(img_path):

    image = cv2.imread(str(img_path))
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    return cv2.Laplacian(gray, cv2.CV_64F).var()

In [ ]:
def identify_bursts(clusters, filepaths, embeddings):
    """
    clusters: List of indices from the get_connected_clusters function.
    filepaths: List of paths corresponding to the indices.
    embeddings: The numpy array of embeddings.
    """
    to_keep = []
    to_discard = []

    for cluster in clusters:
        
        # Calculate Sharpness for everyone in the cluster
        cluster_paths = [filepaths[i] for i in cluster]
        sharpness_scores = [calculate_sharpness(p) for p in cluster_paths]
        
        # Find the Geometric Centroid
        cluster_embs = embeddings[cluster]
        centroid = np.mean(cluster_embs, axis=0)
        
        # Calculate closeness of each image to the centroid (center) -> higher is more central
        centrality = cosine_similarity(cluster_embs, centroid.reshape(1, -1)).flatten()

        # Pick image that is the sharpest under the top 50% most 'central' images.
        
        # Get indices of the most central images (top half)
        threshold_val = np.median(centrality)
        central_indices = [i for i, c in enumerate(centrality) if c >= threshold_val]
        
        # From those central images, pick the one with the highest sharpness
        best_sub_idx = central_indices[np.argmax([sharpness_scores[i] for i in central_indices])]
        
        winner_idx = cluster[best_sub_idx]
        
        # Track images to keep and to discard
        for idx in cluster:
            if idx == winner_idx:
                to_keep.append(filepaths[idx])
            else:
                to_discard.append(filepaths[idx])
                
    return to_keep, to_discard

In [ ]:
fps_to_keep, fps_to_discard = identify_bursts(
    clusters=clusters, 
    filepaths=absolute_filepaths, 
    embeddings=embs
)

In [ ]:
def check_cluster_consistency(clusters, labels, filepaths):
    """
    Sanity Check: Checks if each cluster (burst) contains only one unique label.
    """
    conflicts = []

    for cluster_indices in clusters:
        # Get labels for this specific cluster from your labels list
        cluster_labels = [labels[i] for i in cluster_indices]
        unique_labels = set(cluster_labels)

        if len(unique_labels) > 1:
            # Store the info for reporting
            conflict_info = {
                "indices": cluster_indices,
                "labels": list(unique_labels),
                "paths": [filepaths[i] for i in cluster_indices]
            }
            conflicts.append(conflict_info)

    # Report results
    if not conflicts:
        print("✅ Check Passed: All clusters have 100% consistent labels.")
    else:
        print(f"❌ Found {len(conflicts)} clusters with label mismatches!")
        for i, c in enumerate(conflicts[:5]): # Show first 5
            print(f"  Conflict {i+1}: Labels {c['labels']} | Paths: {c['paths'][0]} ...")

    return conflicts

# Run it
label_conflicts = check_cluster_consistency(clusters, labels, absolute_filepaths)

In [ ]:
def tag_duplicates_in_fiftyone(dataset, clusters_as_filepaths, to_keep):
    """
    1. Loops through groups to assign a unique ID to every member.
    2. Identifies all images not in 'to_keep' and tags them as 'duplicate'.
    """
    
    # --- Assign Group IDs ---
    # Looping over clusters_as_filepaths just as you suggested
    for i, group in enumerate(clusters_as_filepaths):
        group_id = f"burst_{i}_{uuid.uuid4().hex[:6]}"
        
        # Select the samples in this specific group
        view = dataset.select_by("filepath", [str(f) for f in group])
        
        # Give every member of the group the same ID
        view.set_values("dup_group_id", [group_id] * len(view))

    # --- Tag the Redundant ones ---
    # We find everything in the groups that is NOT the winner
    all_clustered_fps = [str(fp) for group in clusters_as_filepaths for fp in group]
    keep_set = set([str(f) for f in to_keep])
    
    # Filter the ones, that are not to_keep
    to_tag_as_duplicate = [fp for fp in all_clustered_fps if fp not in keep_set]

    # Apply the tag
    if to_tag_as_duplicate:
        dataset.select_by("filepath", to_tag_as_duplicate).tag("duplicate")

    print(f"Assigned IDs to {len(clusters_as_filepaths)} groups.")
    print(f"Tagged {len(to_tag_as_duplicate)} images as 'duplicate'.")

In [ ]:
# tagging
tag_duplicates_in_fiftyone(fo_ds, clusters_as_filepaths, fps_to_keep)

# show in FiftyOne
view = fo_ds.match_tags("duplicate")
session.view

Anleitung - can be deleted later

Use the Lasso tool to select a group of points that are very close together.
Look at the dup_group_id in the sample's metadata.
You will see that the "Cloud" in the 2D plot corresponds exactly to your dup_group_id. This confirms that the mathematical similarity (
>
0.98
>0.98
) and the visual 2D projection (UMAP) both agree these images are redundant.

Open the Embeddings Panel (UMAP/PCA).
* Color by dup_group_id: In the panel settings (gear icon), choose Color by: dup_group_id.
-> What you'll see: You will see many small, multi-colored dots. Groups that are "bursts" will be dots that are physically right on top of each other in the plot.
* Color by Tag: Choose Color by: tags.
-> What you'll see: This is usually the most helpful. You will see "islands" where the center dot is one color (representative) and the surrounding dots in the same spot are another color (duplicate).
* Interactive Selection: Use the Lasso Tool in the 2D plot to circle a small cluster. The sample grid below will filter to show you only those images. You can then verify: "Yes, these are all the same burst of this jaguar."


In [ ]:
fo_dataset.export_metadata_snapshot(EXPORT_DIR)